# 🛢️ Crude Oil Production Forecasting — EDA
**Dataset**: Volve Oilfield Production Data (North Sea, Norway)  
**7 Wells · 2008–2016 · 18,452 daily records**

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('../data/raw/volve_production.csv', parse_dates=['DATEPRD'])
events = pd.read_csv('../data/raw/well_events.csv', parse_dates=['date'])
print(f"Records: {len(df):,} | Wells: {df['WELL_BORE_CODE'].nunique()} | Date range: {df['DATEPRD'].min().date()} → {df['DATEPRD'].max().date()}")
df.head()

## 1. Dataset Overview

In [ ]:
df.info()
print("\nMissing values:")
print(df.isnull().sum())

In [ ]:
df.describe().round(2)

## 2. Field-Level Production Trends

In [ ]:
field = df.groupby('DATEPRD')[['BORE_OIL_VOL','BORE_GAS_VOL','BORE_WAT_VOL']].sum()
field = field.resample('W').mean()

fig, axes = plt.subplots(3, 1, figsize=(14,9), sharex=True)
for ax, col, color, label in zip(axes,
    ['BORE_OIL_VOL','BORE_GAS_VOL','BORE_WAT_VOL'],
    ['steelblue','orange','salmon'],
    ['Oil (Sm³/day)','Gas (Sm³/day)','Water (Sm³/day)']):
    ax.fill_between(field.index, field[col], alpha=0.4, color=color)
    ax.plot(field.index, field[col], color=color, lw=1.5, label=label)
    ax.set_ylabel(label); ax.legend()
plt.suptitle('Volve Field — Weekly Production', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 3. Individual Well Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(14,6))
palette = plt.cm.tab10(np.linspace(0,1,7))
for i, well in enumerate(df['WELL_BORE_CODE'].unique()):
    wdf = df[df['WELL_BORE_CODE']==well].set_index('DATEPRD')['BORE_OIL_VOL']
    wdf = wdf.resample('D').sum().rolling(30).mean()
    ax.plot(wdf.index, wdf, color=palette[i], lw=1.8, label=well.split('-')[-1])
ax.set_ylabel('Oil Volume (Sm³/day)'); ax.legend(ncol=2); ax.set_xlabel('Date')
ax.set_title('Per-Well Oil Production (30-Day MA)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 4. Correlation Analysis

In [ ]:
num_cols = ['BORE_OIL_VOL','BORE_GAS_VOL','BORE_WAT_VOL',
            'WELL_BORE_HOURS','AVG_DOWNHOLE_PRESSURE','GOR','WATER_CUT']
corr = df[num_cols].corr()
plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 5. Reservoir Depletion Indicators

In [ ]:
prod_days = df[df['BORE_OIL_VOL'] > 10].set_index('DATEPRD')
monthly = prod_days.resample('ME')[['GOR','WATER_CUT']].mean()

fig, (ax1, ax2) = plt.subplots(2,1, figsize=(13,7), sharex=True)
ax1.plot(monthly['GOR'], color='orange', lw=2)
ax1.fill_between(monthly.index, monthly['GOR'], alpha=0.2, color='orange')
ax1.set_ylabel('GOR (Sm³/Sm³)'); ax1.set_title('Rising GOR → Gas Cap Expansion')
ax2.plot(monthly['WATER_CUT']*100, color='salmon', lw=2)
ax2.fill_between(monthly.index, monthly['WATER_CUT']*100, alpha=0.2, color='salmon')
ax2.set_ylabel('Water Cut (%)'); ax2.set_title('Rising WCT → Aquifer Influx')
plt.tight_layout(); plt.show()

## 6. Key Takeaways

- Field peaked in **2011** (~5,000 Sm³/day avg)
- Clear **exponential decline** post-2012
- **GOR** and **Water Cut** both rising steadily (reservoir maturation)
- **7 wells** with varied start dates and production profiles
- **90 annotated events** (shutdowns, failures, well tests)